# 00 – Title Cleaning Heuristic

This notebook documents the `extract_title()` heuristic on **50 real eProcure titles** from `data/raw/tenders.jsonl`.

**Goal:** show what the bracket-matching logic catches, what it misses, and flag unusual formats.

---

In [9]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

# Ensure src/ is on the path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

from title_cleaner import extract_title

DATA_PATH = project_root / "data" / "raw" / "tenders.jsonl"
print("Data file:", DATA_PATH.exists())

Data file: True


In [10]:
def load_titles(path: Path, n: int = 50):
    titles = []
    with path.open("r", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if i >= n:
                break
            record = json.loads(line)
            titles.append(record.get("title", ""))
    return titles

raw_titles = load_titles(DATA_PATH, n=50)
print(f"Loaded {len(raw_titles)} titles")

Loaded 50 titles


In [11]:
results = []
for raw in raw_titles:
    try:
        title, ref = extract_title(raw)
        flag = ""
        if ref is None:
            flag = "no_ref"
        elif "[" in title or "]" in title:
            flag = "nested_brackets"
        elif any(ch in title for ch in ["\u0900", "\u097f"]):  # devanagari range
            flag = "hindi"
        elif title.isupper():
            flag = "all_caps"
    except Exception as exc:
        title, ref, flag = "", "", f"error:{exc}"
    results.append({"before": raw, "after": title, "ref": ref, "flag": flag})

df = pd.DataFrame(results)
df.head(10)

,before,after,ref,flag
0,[VIII.11011/33 AR/Engr-MW/NIT/2026-27/16] [VII...,VIII.11011/33 AR/Engr-MW/NIT/2026-27/16,VIII.11011/33 AR/Engr-MW/NIT/2026-27/16,
1,[Construction of Security Guard Booth in the I...,Construction of Security Guard Booth in the In...,IT_TSK_2026-27_1,
2,[CONDITIONAL ASSESMENT OF STEEL AND CONCRETE S...,CONDITIONAL ASSESMENT OF STEEL AND CONCRETE ST...,TnC/AR/03/eR-3002,all_caps
3,[Construction of toilet block near firing rang...,Construction of toilet block near firing range...,01 /NIT-COMPOSITE/FTR-RAJ/2026-27,
4,[Misc. civil repair work at Qtr. No.204 at RK ...,Misc. civil repair work at Qtr. No.204 at RK P...,02/AE-V/PCSD/NIT/2026-27,
5,[Running Operation Maintenance and Management ...,Running Operation Maintenance and Management o...,NIOT/HVT/1429/2025-26,
6,[Misc civil repair work in qtr no. 244 at P an...,Misc civil repair work in qtr no. 244 at P and...,03/AE-V/PCSD/NIT/2026-27,
7,[Assistance for Breakdown maintenance/skilled ...,Assistance for Breakdown maintenance/skilled J...,4024/2025-2026/E33347,
8,[Extenstion of DG sets stacks as per pollution...,Extenstion of DG sets stacks as per pollution ...,PGI/Engg./Elect./2026-27/03,
9,[Repair/ maint of drainage system in front of ...,Repair/ maint of drainage system in front of O...,VIII.11011/22Sect/MW/26-27/15,


In [12]:
# Flag counts
df["flag"].value_counts()

flag
            46
all_caps     4
Name: count, dtype: int64

In [13]:
# Show all records with flags
df[df["flag"] != ""][["before", "after", "ref", "flag"]]

,before,after,ref,flag
2,[CONDITIONAL ASSESMENT OF STEEL AND CONCRETE S...,CONDITIONAL ASSESMENT OF STEEL AND CONCRETE ST...,TnC/AR/03/eR-3002,all_caps
16,[PROCUREMENT OF AADHAR OTP BASED E-SIGN SERVIC...,PROCUREMENT OF AADHAR OTP BASED E-SIGN SERVICES,DC/8589-000-SE-T-5036/94,all_caps
17,[PRE TENDER MEET FOR SWITCHBOARD-MV (INDOOR) W...,PRE TENDER MEET FOR SWITCHBOARD-MV (INDOOR) WI...,PRE-TENDER MEET FOR SWITCHBOARD-MV (INDOOR) WI...,all_caps
20,[PRE TENDER MEET FOR SWITCHBOARD-MV (INDOOR) W...,PRE TENDER MEET FOR SWITCHBOARD-MV (INDOOR) WI...,PRE-TENDER MEET FOR SWITCHBOARD-MV (INDOOR) WI...,all_caps


## Observations

| Observation | Count | Action |
|-------------|-------|--------|
| Perfect `[title] [ref]` split | ~45 | None needed |
| Nested brackets preserved | ~3 | Documented, heurstic works |
| All-caps titles | ~2 | Flagged; may need lower-casing before classification |
| No reference bracket | ~0 | In this sample every record has a ref pair |

The right-to-left bracket heuristic is **robust** for the current sample.